# DMMR — suivi du zéro pendant 15 minutes

Suivre les **huit modules ensemble, en gamme fixe 0**, sans correction d'offset. Toutes les nouvelles mesures sont conservées dès le démarrage du flux : pas de chauffe cachée ni d'arrêt à 200 points. Les courants bruts sont en ampères ; les résultats sont présentés en pA et fA.

**Première série : entrées ouvertes, sans câble et sans blindage.** Elle mesure le zéro dans ces conditions, pas le seul offset intrinsèque des modules. Pour comparer ensuite avec un blindage, conserver le même protocole. L'aluminium ne doit pas toucher le contact central ni contaminer l'isolant ; couper et vérifier toute polarisation externe avant de manipuler les entrées.

## Utilisation

Sur le PC **Windows, Python 64 bits**, fermer Explorer et les logiciels CGC. Le port est prérempli à **COM15**. Utiliser **Redémarrer le noyau et tout exécuter** ; la progression s'affiche chaque minute. Le runtime est recherché dans le dossier du notebook, `./dmmr/` ou `~/ESIBD Explorer/plugins/dmmr/`. Si le plugin se trouve ailleurs, renseigner `PLUGIN_DIR`. Dépendances : Jupyter, NumPy, pandas, Matplotlib.

Le test ne modifie ni l'étalonnage ni la configuration persistante. Il sauvegarde les gammes, applique la gamme fixe et vérifie sa relecture. À la fin ou sur interruption, il tente la restauration, puis exige l'arrêt de l'acquisition et la fermeture du port. **Vérifier le bilan d'arrêt.** En cas de DLL bloquée : aucun appel de fermeture concurrent ; vérification locale puis redémarrage du noyau.

Chaque essai crée un dossier distinct contenant `raw.csv`, `report.json` et `native_startup.log`. Les résultats partiels sont conservés. **Aucune mesure simulée n'est produite par ce notebook.** La validation sur le DMMR physique reste nécessaire.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import collections
import csv
import hashlib
import importlib.util
import json
import math
import os
import struct
import sys
import time
import numpy as np
import pandas as pd

COM_PORT = 15
BAUD_RATE = 230400
PLUGIN_DIR = None
OUTPUT_ROOT = Path.cwd() / 'logs' / 'dmmr_zero'

PROTOCOL = {
    'input_condition': 'open_unshielded',  # Ensuite, si nécessaire : 'open_shielded'.
    'ambient_temperature_C': None,        # Seulement si réellement mesurée.
    'notes': '',                         # Heure de mise sous tension, ventilation…
    'range': 0,
    'duration_s': 15 * 60.0,
    'bin_s': 60.0,
    'tail_s': 5 * 60.0,
    'stall_timeout_s': 30.0,              # Arrêt si un module ne fournit plus de mesure valide.
    'io_timeout_s': 5.0,
}

## Ce que permet le certificat

Les pages 3–10 indiquent **200 points, “input open & screened”**, pour chacune des cinq gammes. Elles ne donnent ni cadence, ni durée de stabilisation, ni température. La colonne **“Mean Deviation” n'est pas définie** : elle ne sera assimilée ni à une tolérance, ni à un écart-type, ni à l'incertitude de la moyenne.

- Le P/N **132310**, gamme 0, avait déjà **+534,305 fA** à l'usine.
- Pour le **132303**, l'ordre des bruits et offsets du tableau zéro semble inversé par rapport aux sept autres modules. Les valeurs ci-dessous restent dans **l'ordre imprimé** ; aucun écart au certificat n'est calculé pour ce module avant clarification CGC.
- Sur la page 8 (**132308**), les lignes 2 et 3 du tableau « 1 µA » reprennent des valeurs du tableau « 10 nA » avec un exposant −8. Ce tableau n'est pas utilisé ici ; demander confirmation à CGC.

Ce document est un relevé de tests, **pas une spécification d'acceptation**. La [fiche publique DMM-1PA](https://www.cgc-instruments.com/en/Products/Experiment-Control/DMMR/DMM-1PA) annonce environ 30 fA RMS et 100 fA crête-à-crête : ce sont des indications de bruit, pas une tolérance d'offset. Sans courant de référence, aucun contrôle du gain ou de la linéarité n'est possible. Le rapport indique donc toujours « conformité non évaluable : limites d'acceptation non fournies ».

Transcription du PDF `test_certificat_DMMR.pdf`, SHA-256 conservé ci-dessous. Le rapprochement est fait par **P/N lu sur le matériel**, jamais en supposant que l'adresse 0 correspond au premier certificat. Les identifiants de gamme 0–4 sont conservés : aucune pleine échelle non documentée n'est inventée.

In [ ]:
# Transcription visuellement vérifiée des pages 3–10, dans l'ordre imprimé.
CERTIFICATE_SHA256 = '72a36960479b9c94c18138921a5e135538bdd670014b47824760d89a54614189'
CERTIFICATE_DATE = '2025-02-18'
# P/N : (moyennes en A pour les gammes 0..4, colonne « Mean Deviation » en A).
CERTIFICATE = {
    132303: ([-1.269036e-10, -3.490125e-12, -3.894800e-13, -1.974750e-13, -8.585000e-15],
             [2.4e-11, 2.5e-12, 2.4e-13, 3.4e-14, 1.1e-14]),
    132304: ([+5.731000e-14, -1.005750e-13, -9.767500e-14, -8.240765e-12, -1.291944e-10],
             [1.0e-14, 3.6e-14, 2.5e-13, 2.3e-12, 2.4e-11]),
    132305: ([-1.460000e-14, +1.758000e-14, +2.006405e-12, +1.635747e-11, -1.038038e-10],
             [1.1e-14, 3.7e-14, 2.5e-13, 2.5e-12, 2.4e-11]),
    132306: ([+1.476500e-14, +1.048750e-13, +1.387500e-14, +6.466600e-13, -7.309525e-11],
             [1.1e-14, 3.3e-14, 2.4e-13, 2.6e-12, 2.5e-11]),
    132307: ([+7.665000e-15, -1.090900e-13, -7.440000e-15, -4.662260e-12, -8.422875e-11],
             [1.0e-14, 3.4e-14, 2.6e-13, 2.3e-12, 2.3e-11]),
    132308: ([+1.185000e-14, +6.838500e-14, +7.376200e-13, +9.401575e-12, -1.959550e-11],
             [1.1e-14, 3.5e-14, 2.8e-13, 2.3e-12, 2.7e-11]),
    132309: ([+7.040000e-15, +1.646200e-13, +5.774650e-13, +9.554995e-12, -2.145025e-11],
             [1.1e-14, 3.6e-14, 2.7e-13, 2.2e-12, 2.4e-11]),
    132310: ([+5.343050e-13, +6.315450e-13, +2.713625e-12, +1.637357e-11, -4.442775e-11],
             [1.4e-14, 3.6e-14, 2.4e-13, 2.5e-12, 2.6e-11]),
}
AMBIGUOUS_CERTIFICATE_PNS = {132303}
certificate_table = pd.DataFrame([
    {'P/N': pn, 'page_PDF': pn - 132300, 'gamme': r,
     'offset_CGC_pA': means[r] * 1e12,
     'Mean_Deviation_CGC_fA': deviations[r] * 1e15,
     'ordre_a_confirmer': pn in AMBIGUOUS_CERTIFICATE_PNS}
    for pn, (means, deviations) in CERTIFICATE.items() for r in range(5)
])

## Acquisition et erreurs de communication

Le flux automatique CGC fournit **adresse, courant, gamme effective et horodatage** dans chaque trame. Aucun polling manuel n'est envoyé pendant la mesure. Des valeurs identiques avec des horodatages nouveaux sont conservées ; les horodatages répétés ou décroissants sont exclus des statistiques mais restent dans le CSV. Aucune indépendance statistique des conversions n'est supposée.

Une gamme différente de celle demandée, un courant non fini, une erreur du parseur ou 30 secondes sans nouvelle mesure valide pour l'un des modules arrêtent l'essai. Les 15 minutes sont un choix de protocole, **pas une prescription CGC ni une garantie de stabilisation**.

L'essai précédent a échoué sur `initial range : statut -10`. Ce statut signifie que la DLL n'a pas reçu le caractère de commande attendu. La trace du plugin montre qu'une lecture suivante peut fonctionner, sans établir la cause de la réponse manquante. Pour une lecture de gamme seulement, le notebook autorise **une seule relecture après un −10**, précédée d'une réponse contrôleur valide et d'un état `ST_ON`. L'erreur et les deux tentatives sont enregistrées avec l'adresse. Une autre erreur, un second échec ou une DLL bloquée ne sont jamais ignorés. Toutes les gammes initiales doivent être connues avant le premier changement.

La capture native couvre la préparation et l'activation du flux, puis s'arrête avant le suivi continu. Les délais série ne sont pas augmentés et aucune pause arbitraire n'est ajoutée. Le plugin Explorer et son runtime ne sont pas modifiés par ce notebook.

In [ ]:
def load_driver(plugin_dir):
    """Charge exclusivement le runtime livré avec ce dossier DMMR."""
    root = Path(plugin_dir).resolve() / 'vendor' / 'runtime'
    name = '_esibd_bundled_dmmr_zero_check'
    if not (root / '__init__.py').is_file():
        raise ModuleNotFoundError(f'Runtime DMMR absent : {root}')
    if name in sys.modules and Path(sys.modules[name].__file__).resolve().parent != root:
        raise RuntimeError('Un autre runtime DMMR est déjà chargé : redémarrer le noyau.')
    if name not in sys.modules:
        spec = importlib.util.spec_from_file_location(
            name, root / '__init__.py', submodule_search_locations=[str(root)])
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        try:
            spec.loader.exec_module(module)
        except BaseException:
            for key in list(sys.modules):
                if key == name or key.startswith(name + '.'):
                    del sys.modules[key]
            raise
    return sys.modules[name].DMMR


def checked(device, action, result):
    status, values = (result[0], result[1:]) if isinstance(result, tuple) else (result, ())
    if status != device.NO_ERR:
        raise RuntimeError(f'{action} : statut {status}')
    return values


def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, np.generic):
        return clean_json(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def save_report(output_dir, report):
    path = Path(output_dir) / 'report.json'
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(clean_json(report), indent=2, ensure_ascii=False,
                                    allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def stop_stream(device, timeout):
    checked(device, 'automatic OFF', device.set_automatic_current(False, timeout_s=timeout))
    (enabled,) = checked(device, 'automatic OFF readback', device.get_automatic_current(timeout_s=timeout))
    if enabled is not False:
        raise RuntimeError('Le flux automatique reste activé.')


def verify_controller(device, timeout):
    state, label = checked(device, 'controller state', device.get_state(timeout_s=timeout))
    value = int(state, 16) if isinstance(state, str) else int(state)
    if value != 0:  # COM_DMMR_8_ST_ON = 0, d'après le header CGC.
        raise RuntimeError(f'État contrôleur inattendu : {state} ({label}).')


def read_range(device, address, timeout, events, action):
    """Une seule relecture d'une requête sans effet de bord, uniquement après -10.

    Une réponse contrôleur valide sépare les requêtes : ne pas accepter aveuglément
    une éventuelle réponse tardive à la première requête. Aucun timeout augmenté,
    aucune réécriture, aucune reprise après blocage de la DLL.
    """
    for attempt in (1, 2):
        result = device.get_module_meas_range(address, timeout_s=timeout)
        status = result[0]
        events.append({'action': action, 'address': address, 'attempt': attempt,
                       'status': status, 'utc': datetime.now(timezone.utc).isoformat(),
                       'range': result[1] if status == device.NO_ERR else None,
                       'auto': result[2] if status == device.NO_ERR else None})
        if status == -10 and attempt == 1 and not getattr(device, '_transport_poisoned', False):
            print(f'{action}, module {address} : -10 ; contrôle du contrôleur puis une relecture.', flush=True)
            verify_controller(device, timeout)
            continue
        actual, auto = checked(device, f'{action}, module {address}, lecture {attempt}', result)
        if type(actual) is not int or actual not in range(5) or type(auto) is not bool:
            raise RuntimeError(f'{action}, module {address} : réponse de gamme invalide ({actual}, {auto}).')
        return actual, auto


def get_stream_frame(device, timeout):
    # GetCurent peut masquer une erreur du parseur automatique de la DLL.
    parsed = device._call_locked_with_timeout(device.check_auto_input, timeout, 'check_auto_input')
    checked(device, 'parse automatic input', parsed)
    return device.get_current(timeout_s=timeout)


def flush_frames(device, timeout, limit=10000):
    # Après arrêt du flux seulement. Aucune purge pendant la mesure.
    for count in range(limit):
        result = get_stream_frame(device, timeout)
        if result[0] == device.NO_DATA:
            return count
        checked(device, 'purge FIFO', result)
    raise RuntimeError('La FIFO ne se vide pas après arrêt du flux.')


def configure_range(device, addresses, requested, timeout, events, changed):
    for address in addresses:
        changed.add(address)  # Même un ACK perdu peut avoir modifié le matériel.
        checked(device, f'set manual, module {address}', device.set_module_auto_range(
            address, False, timeout_s=timeout))
        checked(device, f'set range, module {address}', device.set_module_meas_range(
            address, requested, timeout_s=timeout))
        actual, auto = read_range(device, address, timeout, events, 'range readback')
        if auto is not False or actual != requested:
            raise RuntimeError(f'Module {address} : gamme fixe {requested} non confirmée ({actual}, auto={auto}).')


def finish_capture(device, output_dir, report, timeout):
    diagnostic = report['diagnostics']
    if diagnostic.get('ended'):
        return
    diagnostic['ended'] = True
    if getattr(device, '_transport_poisoned', False):
        text = 'DLL bloquée : capture non fermée ; aucun appel concurrent. Voir le fichier partiel indiqué au démarrage.'
    else:
        try:
            text = device.end_startup_diagnostics(timeout_s=timeout)
        except Exception as exc:
            text = f'Capture native indisponible : {type(exc).__name__}: {exc}'
    (Path(output_dir) / 'native_startup.log').write_text(
        diagnostic.get('start', '') + '\n' + text + '\n', encoding='utf-8')
    diagnostic['file'] = 'native_startup.log'


RAW_COLUMNS = ['phase', 'address', 'product_no', 'requested_range', 'actual_range',
               'host_utc', 'host_elapsed_s', 'device_time_s', 'current_A',
               'status', 'selected', 'reason']


def collect_run(device, modules, cfg, writer, raw_file, report, output_dir,
                clock=time.monotonic, sleep=time.sleep):
    """Suivi continu de durée fixée ; aucune chauffe masquée, aucun quota de points."""
    start = clock()
    counts = {a: 0 for a in modules}
    last_time, last_valid = {}, {a: start for a in modules}
    reasons = collections.Counter()
    info = {'points_by_address': counts, 'duration_s': 0.0, 'empty_polls': 0,
            'frame_reasons': {}, 'complete': False}
    report['acquisition'] = info
    next_progress = cfg['bin_s']
    try:
        checked(device, 'automatic ON', device.set_automatic_current(True, timeout_s=cfg['io_timeout_s']))
        # Capturer l'ACK de démarrage, mais pas 15 minutes de trace native.
        finish_capture(device, output_dir, report, cfg['io_timeout_s'])
        if getattr(device, '_transport_poisoned', False):
            raise RuntimeError('DLL bloquée pendant la capture native.')
        while clock() - start < cfg['duration_s']:
            stale = [a for a in modules if clock() - last_valid[a] >= cfg['stall_timeout_s']]
            if stale:
                raise TimeoutError(f'Aucune nouvelle mesure valide depuis {cfg["stall_timeout_s"]:g} s : modules {stale}.')
            status, address, current, actual, device_time = get_stream_frame(device, cfg['io_timeout_s'])
            now = clock()
            if status == device.NO_DATA:
                info['empty_polls'] += 1
                sleep(0.01)
                continue
            reason, fatal = 'selected', False
            if status != device.NO_ERR:
                reason, fatal = f'status_{status}', True
            elif address not in modules:
                reason, fatal = 'unknown_address', True
            elif not (math.isfinite(current) and math.isfinite(device_time)):
                reason, fatal = 'nonfinite', True
            elif type(actual) is not int or actual not in range(5):
                reason, fatal = 'invalid_range', True
            elif actual != cfg['range']:
                reason, fatal = 'wrong_range', True
            elif device_time <= last_time.get(address, -math.inf):
                reason = 'timestamp_not_increasing'
            elif now - start >= cfg['duration_s']:
                reason = 'after_duration'
            else:
                last_time[address] = device_time
                last_valid[address] = now
                counts[address] += 1
            selected = reason == 'selected'
            reasons[reason] += 1
            valid_payload = status == device.NO_ERR
            writer.writerow({
                'phase': 'continuous', 'address': address if valid_payload else '',
                'product_no': modules.get(address, {}).get('product_no', '') if valid_payload else '',
                'requested_range': cfg['range'], 'actual_range': actual if valid_payload else '',
                'host_utc': datetime.now(timezone.utc).isoformat(), 'host_elapsed_s': now - start,
                'device_time_s': device_time if valid_payload else '', 'current_A': current if valid_payload else '',
                'status': status, 'selected': int(selected), 'reason': reason})
            raw_file.flush()
            if fatal:
                raise RuntimeError(f'Acquisition, module {address} : {reason}')
            if now - start >= next_progress:
                info['duration_s'] = now - start
                info['frame_reasons'] = dict(reasons)
                save_report(output_dir, report)
                print(f'{(now-start)/60:.1f}/{cfg["duration_s"]/60:g} min — points par module : {counts}', flush=True)
                next_progress += cfg['bin_s']
        stale = [a for a in modules if counts[a] < 2 or clock() - last_valid[a] >= cfg['stall_timeout_s']]
        if stale:
            raise TimeoutError(f'Acquisition incomplète pour les modules {stale}.')
        info['complete'] = True
    finally:
        info['duration_s'] = clock() - start
        info['frame_reasons'] = dict(reasons)


def restore_and_shutdown(device, original, timeout, events=None):
    result = {'range_restoration_needed': bool(original), 'ranges_restored': False,
              'shutdown_confirmed': False, 'errors': []}
    events = [] if events is None else events
    if getattr(device, '_transport_poisoned', False):
        result['errors'].append('DLL bloquée : aucun nouvel appel. Arrêt non confirmé ; redémarrer le noyau après vérification locale.')
        return result
    try:
        stop_stream(device, timeout)
        for address, (meas_range, auto) in original.items():
            checked(device, f'restore manual, module {address}', device.set_module_auto_range(address, False, timeout_s=timeout))
            checked(device, f'restore range, module {address}', device.set_module_meas_range(address, meas_range, timeout_s=timeout))
            checked(device, f'restore auto, module {address}', device.set_module_auto_range(address, auto, timeout_s=timeout))
            actual, actual_auto = read_range(device, address, timeout, events, 'restore readback')
            if actual_auto != auto or (not auto and actual != meas_range):
                raise RuntimeError(f'Restauration non confirmée pour le module {address}.')
        result['ranges_restored'] = True
    except (Exception, KeyboardInterrupt) as exc:
        result['errors'].append(f'Restauration : {type(exc).__name__}: {exc}')
    if not getattr(device, '_transport_poisoned', False):
        try:
            result['shutdown_confirmed'] = device.shutdown(timeout_s=timeout) is True
            if not result['shutdown_confirmed']:
                result['errors'].append('Le driver ne confirme pas OFF + fermeture du port.')
        except (Exception, KeyboardInterrupt) as exc:
            result['errors'].append(f'Arrêt : {type(exc).__name__}: {exc}')
    else:
        result['errors'].append('Transport bloqué pendant la restauration ; aucun appel de fermeture concurrent.')
    return result


def validate_protocol(cfg):
    if cfg['input_condition'] not in ('open_unshielded', 'open_shielded'):
        raise ValueError('input_condition doit être open_unshielded ou open_shielded.')
    if isinstance(cfg['range'], bool) or not isinstance(cfg['range'], int) or cfg['range'] not in range(5):
        raise ValueError('range doit être un entier de 0 à 4.')
    for key in ('duration_s', 'bin_s', 'tail_s', 'stall_timeout_s', 'io_timeout_s'):
        value = cfg[key]
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:
            raise ValueError(f'{key} doit être un nombre fini strictement positif.')
    if any(cfg[key] > cfg['duration_s'] for key in ('bin_s', 'tail_s', 'stall_timeout_s')):
        raise ValueError('Les fenêtres et le délai sans données ne doivent pas dépasser la durée de mesure.')


def run_zero_check(device, output_dir, cfg, provenance=None, clock=time.monotonic, sleep=time.sleep):
    validate_protocol(cfg)
    if getattr(device, '_transport_poisoned', False) or getattr(device, 'connected', False):
        raise RuntimeError('Utiliser une instance déconnectée et non bloquée ; aucun réessai automatique de la série.')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    report = {'schema': 2, 'started_utc': datetime.now(timezone.utc).isoformat(),
              'protocol': dict(cfg), 'provenance': provenance or {}, 'modules': {},
              'certificate_sha256': CERTIFICATE_SHA256, 'certificate_date': CERTIFICATE_DATE,
              'certificate_values': CERTIFICATE, 'certificate_ambiguous_pns': sorted(AMBIGUOUS_CERTIFICATE_PNS),
              'conformity': 'not_assessable_no_acceptance_limits', 'status': 'running',
              'initial_ranges': {}, 'range_reads': [], 'diagnostics': {},
              'acquisition': None, 'error': None, 'cleanup': None}
    original, changed = {}, set()
    connection_attempted = False
    save_report(output_dir, report)
    try:
        with (output_dir / 'raw.csv').open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=RAW_COLUMNS)
            writer.writeheader()
            stream.flush()
            try:
                report['diagnostics']['start'] = device.begin_startup_diagnostics(timeout_s=cfg['io_timeout_s'])
                connection_attempted = True
                if device.connect(timeout_s=cfg['io_timeout_s']) is not True:
                    raise RuntimeError('Connexion non confirmée.')
                stop_stream(device, cfg['io_timeout_s'])
                checked(device, 'enable', device.set_enable(True, timeout_s=cfg['io_timeout_s']))
                (enabled,) = checked(device, 'enable readback', device.get_enable(timeout_s=cfg['io_timeout_s']))
                if enabled is not True:
                    raise RuntimeError('Activation des modules non confirmée.')
                modules = device.initialize(timeout_s=cfg['io_timeout_s'], persist_scan=False)
                report['modules'] = modules
                pns = [info.get('product_no') for info in modules.values()]
                if len(modules) != 8 or len(set(pns)) != 8 or set(pns) != set(CERTIFICATE):
                    raise RuntimeError(f'Identités différentes du certificat : {pns}. Aucun rapprochement par position.')
                report['controller'] = device.get_product_info(timeout_s=cfg['io_timeout_s'])
                verify_controller(device, cfg['io_timeout_s'])
                for address in modules:
                    original[address] = read_range(device, address, cfg['io_timeout_s'], report['range_reads'], 'initial range')
                    report['initial_ranges'][address] = original[address]
                # Ne rien modifier tant que toutes les gammes initiales ne sont pas connues.
                configure_range(device, modules, cfg['range'], cfg['io_timeout_s'], report['range_reads'], changed)
                verify_controller(device, cfg['io_timeout_s'])
                report['frames_flushed_before_run'] = flush_frames(device, cfg['io_timeout_s'])
                save_report(output_dir, report)
                print(f'Acquisition : {cfg["duration_s"]/60:g} min, gamme fixe {cfg["range"]}, huit modules.', flush=True)
                collect_run(device, modules, cfg, writer, stream, report, output_dir, clock=clock, sleep=sleep)
                stop_stream(device, cfg['io_timeout_s'])
                verify_controller(device, cfg['io_timeout_s'])
                report['status'] = 'acquisition_complete'
            except (Exception, KeyboardInterrupt) as exc:
                report['status'] = 'interrupted' if isinstance(exc, KeyboardInterrupt) else 'failed'
                report['error'] = f'{type(exc).__name__}: {exc}'
                print(f'SÉRIE INCOMPLÈTE : {report["error"]}', flush=True)
            finally:
                if connection_attempted and (getattr(device, 'connected', False) or getattr(device, '_transport_poisoned', False)):
                    report['cleanup'] = restore_and_shutdown(device, {a: original[a] for a in changed},
                                                             cfg['io_timeout_s'], report['range_reads'])
                else:
                    report['cleanup'] = {'ranges_restored': not changed, 'shutdown_confirmed': False,
                                         'errors': ['Aucune liaison disponible pour confirmer l’arrêt.']}
                finish_capture(device, output_dir, report, cfg['io_timeout_s'])
    finally:
        report['finished_utc'] = datetime.now(timezone.utc).isoformat()
        if report['status'] == 'acquisition_complete':
            report['status'] = 'complete' if report['cleanup'] and not report['cleanup']['errors'] else 'cleanup_failed'
        save_report(output_dir, report)
    print(f'Résultat : {report["status"]} — {output_dir}')
    if report['cleanup']['errors']:
        print('ATTENTION :', '\n'.join(report['cleanup']['errors']))
    return report


def sample_statistics(group, report):
    pn, actual = int(group['product_no'].iloc[0]), int(group['actual_range'].iloc[0])
    x = group['current_A'].to_numpy(dtype=float)
    t = group['device_time_s'].to_numpy(dtype=float)
    mean = float(x.mean())
    reference = report['certificate_values'][str(pn)][0][actual]
    ambiguous = pn in report['certificate_ambiguous_pns']
    return {
        'adresse': int(group['address'].iloc[0]), 'P/N': pn, 'gamme': actual, 'N': len(x),
        'moyenne_pA': mean * 1e12,
        'ecart_type_fA': float(np.std(x, ddof=1)) * 1e15 if len(x) > 1 else None,
        'ecart_absolu_moyen_fA': float(np.mean(np.abs(x - mean))) * 1e15,
        'crete_a_crete_fA': float(np.ptp(x)) * 1e15,
        'derive_fA_min': float(np.polyfit(t - t[0], x - mean, 1)[0]) * 60e15 if len(t) > 1 and np.ptp(t) > 0 else None,
        'duree_observee_s': float(np.ptp(t)),
        'debut_hote_s': float(group['host_elapsed_s'].min()),
        'fin_hote_s': float(group['host_elapsed_s'].max()),
        'pas_median_s': float(np.median(np.diff(t))) if len(t) > 1 else None,
        'pas_max_s': float(np.max(np.diff(t))) if len(t) > 1 else None,
        'offset_CGC_pA': reference * 1e12,
        'Mean_Deviation_CGC_fA': report['certificate_values'][str(pn)][1][actual] * 1e15,
        'ecart_au_certificat_pA': None if ambiguous else (mean - reference) * 1e12,
        'comparaison': 'ordre CGC à confirmer' if ambiguous else ('entrée non blindée' if report['protocol']['input_condition'] == 'open_unshielded' else 'limites CGC non fournies'),
    }


def analyse_run(output_dir):
    output_dir = Path(output_dir)
    report = json.loads((output_dir / 'report.json').read_text(encoding='utf-8'))
    if report.get('schema') != 2:
        raise ValueError('Ancien protocole : ne pas mélanger les phases du balayage avec le suivi continu.')
    raw = pd.read_csv(output_dir / 'raw.csv', float_precision='round_trip')
    samples = raw.loc[(raw['selected'] == 1) & (raw['status'] == 0)].copy()
    summary, binned, tail = [], [], []
    cfg = report['protocol']
    for _, group in samples.groupby(['address', 'product_no', 'actual_range'], sort=True):
        summary.append(sample_statistics(group, report))
        bins = np.floor(group['host_elapsed_s'] / cfg['bin_s']).astype(int)
        for index, chunk in group.groupby(bins):
            row = sample_statistics(chunk, report)
            row.update(debut_s=float(index * cfg['bin_s']),
                       fin_s=float(min((index + 1) * cfg['bin_s'], cfg['duration_s'])))
            binned.append(row)
        # Pas de « cinq dernières minutes » sur une acquisition interrompue.
        if (report.get('acquisition') or {}).get('complete'):
            chunk = group.loc[group['host_elapsed_s'] >= cfg['duration_s'] - cfg['tail_s']]
            if len(chunk):
                row = sample_statistics(chunk, report)
                row['stabilite'] = 'à examiner sur la trace et les moyennes par minute'
                tail.append(row)
    report['summary'] = summary
    report['time_bins'] = binned
    report['tail_summary'] = tail
    report['frame_reasons'] = raw['reason'].value_counts().to_dict()
    save_report(output_dir, report)
    return raw, pd.DataFrame(summary), report


def plot_run(raw, report):
    import matplotlib.pyplot as plt
    frame = raw.loc[(raw['status'] == 0) & (raw['selected'] == 1)].copy()
    fig, axes = plt.subplots(4, 2, figsize=(12, 11), layout='constrained')
    modules = sorted((int(a), info) for a, info in report['modules'].items())
    for ax, (address, info) in zip(axes.flat, modules):
        data = frame.loc[frame['address'] == address]
        ax.plot(data['host_elapsed_s'] / 60, data['current_A'] * 1e12,
                '.', ms=2, alpha=.3, color='0.5', label='Mesures')
        bins = [row for row in report.get('time_bins', []) if row['adresse'] == address]
        if bins:
            x = [(row['debut_s'] + row['fin_s']) / 120 for row in bins]
            y = [row['moyenne_pA'] for row in bins]
            err = [np.nan if row['ecart_type_fA'] is None else row['ecart_type_fA'] / 1000 for row in bins]
            ax.errorbar(x, y, yerr=err, fmt='o-', ms=3, lw=1, capsize=2,
                        color='C0', label='Moyenne par minute ± écart-type')
        ax.set_title(f'P/N {info["product_no"]} · adresse {address}', fontsize=10)
        ax.set_ylabel('Courant (pA)')
        ax.ticklabel_format(axis='y', style='plain', useOffset=False)
        ax.set_xlim(0, report['protocol']['duration_s'] / 60)
        ax.grid(alpha=.2)
    for ax in axes.flat[len(modules):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel('Temps depuis démarrage du flux (min)')
    condition = 'entrée ouverte non blindée' if report['protocol']['input_condition'] == 'open_unshielded' else 'entrée ouverte blindée'
    partial = '' if (report.get('acquisition') or {}).get('complete') else ' — série partielle'
    fig.suptitle(f'DMMR · gamme fixe {report["protocol"]["range"]} · {condition}{partial}\n'
                 'Gris : mesures · Bleu : moyenne par minute ± écart-type', fontsize=11)
    return fig


## Acquisition réelle

Ne pas toucher les entrées pendant l'essai. `complete` signifie que la mesure, la restauration et l'arrêt ont réussi — **pas que l'appareil est conforme**. Une série en échec ne doit pas être confondue avec les références du certificat.

In [ ]:
if os.name != 'nt' or struct.calcsize('P') != 8:
    raise RuntimeError('Acquisition matérielle : Windows avec Python 64 bits requis.')
if isinstance(COM_PORT, bool) or not isinstance(COM_PORT, int) or not 1 <= COM_PORT <= 255:
    raise ValueError('Renseigner COM_PORT avec le numéro du port (1–255).')
validate_protocol(PROTOCOL)

previous = globals().get('_zero_device')
if previous is not None and (getattr(previous, '_transport_poisoned', False) or getattr(previous, 'connected', False)):
    raise RuntimeError('La précédente connexion n’est pas libérée. Vérifier l’arrêt local ; ne pas créer une connexion concurrente.')

if PLUGIN_DIR is None:
    candidates = [Path.cwd(), Path.cwd() / 'dmmr', Path.home() / 'ESIBD Explorer' / 'plugins' / 'dmmr']
    PLUGIN_DIR = next((p for p in candidates if (p / 'dmmr_plugin.py').is_file()), None)
if PLUGIN_DIR is None:
    raise FileNotFoundError('Dossier dmmr introuvable : renseigner PLUGIN_DIR dans les réglages.')
PLUGIN_DIR = Path(PLUGIN_DIR).resolve()
DMMR = load_driver(PLUGIN_DIR)
paths = ['vendor/runtime/dmmr/dmmr.py', 'vendor/runtime/dmmr/dmmr_base.py',
         'vendor/runtime/_driver_common.py', 'vendor/runtime/dmmr/vendor/x64/COM-DMMR-8.dll']
provenance = {'plugin_dir': str(PLUGIN_DIR), 'python': sys.version, 'com': COM_PORT, 'baud_rate': BAUD_RATE,
              'sha256': {name: hashlib.sha256((PLUGIN_DIR / name).read_bytes()).hexdigest() for name in paths}}
RUN_DIR = OUTPUT_ROOT / (datetime.now().strftime('%Y%m%d_%H%M%S_%f') + '_' + PROTOCOL['input_condition'])
_zero_device = DMMR('dmmr_zero_check', com=COM_PORT, baudrate=BAUD_RATE,
                    process_backend=False, log_dir=OUTPUT_ROOT)
report = run_zero_check(_zero_device, RUN_DIR, PROTOCOL, provenance=provenance)

## Résultats mesurés

- **Moyenne** : courant moyen, sans correction d'offset.
- **Écart-type** : dispersion autour de cette moyenne, avec correction `ddof=1`. Le rapport conserve aussi l'écart absolu moyen et la valeur crête-à-crête.
- **Évolution temporelle** : trace brute, moyenne et écart-type par minute. L'écart-type global inclut les variations lentes ; ne pas l'assimiler au seul bruit instrumental.
- **Cinq dernières minutes** : tableau séparé, uniquement si toute la durée a été acquise. **Vérifier visuellement la stabilité avant d'utiliser cette moyenne comme estimation du zéro.** Aucune stabilité n'est déclarée automatiquement.
- **Dérive** : pente linéaire descriptive en fA/min, pas critère d'acceptation.

Les barres des figures sont des **écarts-types, pas des incertitudes sur la moyenne**. Pas de division par √N : autocorrélation et bande passante ne sont pas établies. Les valeurs CGC sont uniquement des références, distinguées des mesures. Le tracé ne force pas l'axe vertical à inclure ces références, afin de laisser visibles les petites variations temporelles.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

raw, summary, report = analyse_run(RUN_DIR)
print('État de la série :', report['status'])
print('Conformité : non évaluable sans limites CGC et conditions comparables.')
if report['error']:
    print('ERREUR :', report['error'])
print('Arrêt :', report['cleanup'])
failures = [event for event in report.get('range_reads', []) if event['status'] != 0]
if failures:
    print('Erreurs de lecture de gamme conservées dans le rapport :')
    display(pd.DataFrame(failures)[['action', 'address', 'attempt', 'status']])

columns = ['P/N', 'adresse', 'gamme', 'N', 'moyenne_pA', 'ecart_type_fA',
           'ecart_absolu_moyen_fA', 'derive_fA_min', 'duree_observee_s']
if len(summary):
    print('MESURES — ensemble de la série :')
    display(summary[columns])
    tail = pd.DataFrame(report['tail_summary'])
    if len(tail):
        print('MESURES — cinq dernières minutes. Stabilité à vérifier sur les courbes :')
        display(tail[columns])
        print('Références CGC comparées à cette fenêtre finale — pas de tolérances :')
        display(tail[['P/N', 'moyenne_pA', 'offset_CGC_pA', 'ecart_au_certificat_pA', 'comparaison']])
    else:
        print('Série partielle : pas de bilan des cinq dernières minutes prévues.')
    fig = plot_run(raw, report)
    plt.show()
    plt.close(fig)
    print('Moyennes et écarts-types par minute conservés dans report.json → time_bins.')
else:
    print('AUCUNE MESURE VALIDE. Ne pas interpréter les références CGC comme des résultats.')
    print('Consulter report.json et native_startup.log avant un nouvel essai.')
display(raw['reason'].value_counts().rename('Nombre de trames'))
print('Données et rapport :', RUN_DIR.resolve())

## Pour conclure avec CGC

Transmettre le rapport, les traces et les conditions d'entrée. Demander : **tolérance d'offset par gamme**, définition de « Mean Deviation », cadence/filtrage et stabilisation du test d'usine, confirmation du tableau zéro du P/N 132303. Si la série à entrée ouverte présente un écart, la répétition blindée est nécessaire avant de l'attribuer au module. Si l'écart persiste, CGC pourra définir le contrôle suivant ; ne pas retoucher l'étalonnage pour le masquer.